# Habitat and species distribution modelling: grassland richness in the Alps and Carpathians
#an ultimate goal of the G4B project /grass4b.com/

Authors: G4Bs

Purpose:
1) Presentation of the results at WSL
2) A paper on similar topic

Content:
1. Intro
2. Response variable: grassland habitat
3. Area of interest for the model: Alps&Carps and grassland areas
4. Explanatory variables
  *   4.1 Sentinel-2
  *   4.2 Terrain, climate, soils, solar radiation...

This code is inspired on the SDM workflow available at the Google Developer page https://developers.google.com/earth-engine/tutorials/community/species-distribution-modeling

We modified the code to do the tiling and ensemble modelling across the the Alps and Carpathians.

For more information or the giving the access to the stored data please contact: robert.pazur@wsl.ch

In [ ]:
import ee
# Trigger the authentication flow.
ee.Authenticate()
# Initialize the library.
ee.Initialize(project='g4bproject')

In [ ]:

# Import libraries
import geemap
import geemap.colormaps as cm
import pandas as pd, geopandas as gpd
import numpy as np, matplotlib.pyplot as plt
import os, requests, math, random
from ipyleaflet import TileLayer
from statsmodels.stats.outliers_influence import variance_inflation_factor
from tqdm import tqdm


## 2 Grassland richness dataset
We used the grassland richness dataset obtained from the European Vegetation Survey) that was filtered for Alps and Carpathians.

Furthermore, we filtered the data by dates, sampled area (this is especially important) and habitat type.

In [ ]:
#botdata_raw = (ee.FeatureCollection("projects/g4bproject/assets/botanicalData/EVAdataDL_outcome20250830"))
botdata_raw = (ee.FeatureCollection("projects/g4bproject/assets/botanicalData/species_richnessEVAAPVVCombined_v20260304"))
#get the columns
# Get the first feature to extract property names
first_feature = botdata_raw.first()
column_names = first_feature.propertyNames().getInfo()

print("Available column names:")
print(column_names)

Available column names:
['area', 'OBJECTID', 'richness', 'Latitude', 'rich_har', 'Longitude', 'years', 'system:index']


###  3 Define the area of interest: the Alps and Carpathians

In [ ]:
aoiGeneral = (ee.FeatureCollection('projects/g4bproject/assets/Alps_Carpathians'))
print(aoiGeneral.size().getInfo())
#get column names
first_feature = aoiGeneral.first()
property_names = first_feature.propertyNames().getInfo()
print("Available column names are:")
print(property_names)

2
Available column names are:
['OBJECTID', 'Shape_Area', 'Id', 'onefield', 'm_range', 'layer', 'system:index']


In [ ]:
# Define the AOI
aoiGeneral = (ee.FeatureCollection('projects/g4bproject/assets/Alps_Carpathians'))
aoiCarpathians = (ee.FeatureCollection('projects/g4bproject/assets/Alps_Carpathians'))
#aoi = (ee.FeatureCollection('projects/g4bproject/assets/Alps'))
# aoi = (ee.FeatureCollection('projects/g4bproject/assets/Alps'))

aoi = (ee.FeatureCollection('projects/g4bproject/assets/Carpathians'))
grid = (ee.FeatureCollection("projects/g4bproject/assets/grids/grid90km") #/grid45km and /grid22_5km possible
        .filterBounds(aoi))
grid45 = (ee.FeatureCollection("projects/g4bproject/assets/grids/grid45km") #/grid45km and /grid22_5km possible
        .filterBounds(aoi))
gridID = grid.aggregate_array('id').getInfo()
grid45ID = grid45.aggregate_array('id').getInfo()

#print(gridID)
lulcAlpsCarps = ee.Image('projects/g4bproject/assets/LC_con_RF').clip(aoi)
grassMask = lulcAlpsCarps.eq(5).rename('grassMask').clip(aoi)
vis_params_lulc = {"min":0,"max":9,"palette":["#000000","#cc0303","#cdb400","#235123","#b76124","#92af1f","#f7e174","#2019a4","#aec3d6","#bdbdbd"]}
m = geemap.Map()
m.addLayer(aoi)
m.addLayer(grid)
m.add_labels(
    grid,
    "id",
    font_size="12pt",
    font_color="red",
    font_family="arial",
    font_weight="bold",
)
#m.addLayer(lulcAlpsCarps, vis_params_lulc, 'lulcAlpsCarps')
m.centerObject(aoi, 4)
m


Map(center=[47.376687358003636, 22.726732647051985], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
gridID = grid.aggregate_array('id').getInfo()
print(gridID)

[92, 93, 101, 102, 103, 104, 111, 112, 113, 114, 121, 122, 123, 124, 131, 132, 133, 134, 141, 142, 143, 144, 145, 146, 147, 148, 149, 151, 152, 153, 154, 155, 156, 157, 158, 159, 162, 163, 164, 165, 166, 167, 172, 173, 174, 175, 176, 177, 183, 184, 185, 186, 194, 195, 196]


####  3.1 Further filtering of the data (not necessary)

In [ ]:
for grid_id in gridID:
    # Filter the grid for the current ID
    grid_cell = grid.filter(ee.Filter.eq('id', grid_id))
    print(f"Grid ID: {grid_id}")

##filter to grasslands only
botdata_reduced = grassMask.reduceRegions(
    collection=botdata_raw,
    reducer=ee.Reducer.count(),  # Using count to get the intersection
    scale=10  # Adjust the scale to match your raster resolution
)

botdata = botdata_reduced.filter(ee.Filter.gt('count', 0))
print("Data size:", botdata_raw.size().getInfo())
print("Data size:", botdata.size().getInfo())

Grid ID: 92
Grid ID: 93
Grid ID: 101
Grid ID: 102
Grid ID: 103
Grid ID: 104
Grid ID: 111
Grid ID: 112
Grid ID: 113
Grid ID: 114
Grid ID: 121
Grid ID: 122
Grid ID: 123
Grid ID: 124
Grid ID: 131
Grid ID: 132
Grid ID: 133
Grid ID: 134
Grid ID: 141
Grid ID: 142
Grid ID: 143
Grid ID: 144
Grid ID: 145
Grid ID: 146
Grid ID: 147
Grid ID: 148
Grid ID: 149
Grid ID: 151
Grid ID: 152
Grid ID: 153
Grid ID: 154
Grid ID: 155
Grid ID: 156
Grid ID: 157
Grid ID: 158
Grid ID: 159
Grid ID: 162
Grid ID: 163
Grid ID: 164
Grid ID: 165
Grid ID: 166
Grid ID: 167
Grid ID: 172
Grid ID: 173
Grid ID: 174
Grid ID: 175
Grid ID: 176
Grid ID: 177
Grid ID: 183
Grid ID: 184
Grid ID: 185
Grid ID: 186
Grid ID: 194
Grid ID: 195
Grid ID: 196
Data size: 2053
Data size: 2003


In [ ]:
# Spatial resolution based on the climate data (meters)
grain_size = 100
# Remove data duplicates within the grain size limits (meters)
def remove_duplicates(data, grain_size):
    # Select one occurrence record per pixel at the chosen spatial resolution
    random_raster = ee.Image.random(seed=50).reproject("EPSG:4326", None, grain_size)
    rand_point_vals = random_raster.sampleRegions(
        collection=ee.FeatureCollection(data), geometries=True
    )
    return rand_point_vals.distinct("random")

botdata_sp = remove_duplicates(botdata_raw, grain_size)

# Before selection and after selection
print("Original data size:", botdata_raw.size().getInfo())
print("Final data size:", botdata_sp.size().getInfo())


Original data size: 10261
Final data size: 7824


### 3 Define the area of interest: the Alps and Carpathians

---



## 4.1 Sentinel-2

In [ ]:
#changing satelite band names to make the calculation of indices transparent
originalNames = ['B2','B3','B4', 'B5', 'B6', 'B7','B8','B11','B12']
commonNames = ['blue', 'green', 'red', 'R1', 'R2', 'R3','nir','swir1', 'swir2']
start=2022 #define the start year
end=2024 #define the end year

def geopandas_to_ee(gdf, return_aoi = False):
    samples = geemap.geopandas_to_ee(gdf)
    if return_aoi == True:
        aoi = samples.union().geometry().bounds()
        return samples, aoi
    else:
        return samples

def unique_values(collection, field):
    values = ee.Dictionary(collection.reduceColumns(ee.Reducer.frequencyHistogram(), [field]).get('histogram')).keys()
    return values

def daily_mosaics(imgs):

  def simplifyDate(img):
    d = ee.Date(img.get('system:time_start'))
    day = d.get('day')
    m = d.get('month')
    y = d.get('year')
    simpleDate = ee.Date.fromYMD(y,m,day)
    return img.set('simpleTime',simpleDate.millis())

  imgs = imgs.map(simplifyDate)
  days = unique_values(imgs,'simpleTime')

  def do_mosaic(d):
    d = ee.Number.parse(d)
    d = ee.Date(d)
    t = imgs.filterDate(d,d.advance(1,'day'))
    f = ee.Image(t.first())
    t = t.mosaic()
    t = t.set('system:time_start',d.millis())
    t = t.copyProperties(f)
    return t

  imgs = days.map(do_mosaic)

  return ee.ImageCollection.fromImages(imgs)

def addVariables(img):
  date = img.set({'DATE': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')})
  ndvi = img.normalizedDifference(['nir', 'red']).rename('NDVI')
  ndmi = img.normalizedDifference(['red', 'nir']).rename('NDMI')
  evi = img.expression('2.5 * ((nir - red) / (nir + 6 * red - 7.5 * blue + 1))',{'nir': img.select('nir'), 'red': img.select('red'), 'blue': img.select('blue')}).rename('EVI')
  swir_ratio = img.expression('swir2/swir1',{'swir2': img.select('swir2'), 'swir1': img.select('swir1')}).rename('SWIR_ratio')
  msavi = img.expression('(2 * nir + 1 - sqrt(pow((2 * nir + 1), 2) - 8 * (nir - red)) ) / 2', {'nir': img.select('nir'), 'red': img.select('red')}).rename('MSAVI')
  nbr1 = img.expression('(nir - swir2) / (nir + swir2)',{'nir': img.select('nir'), 'swir2': img.select('swir2')}).rename('NBR1')
  nbr2 = img.expression('(swir1 - swir2) / (swir1 + swir2)',{'swir2': img.select('swir2'), 'swir1': img.select('swir1')}).rename('SWIR_ratio').rename('NBR2')
  # We can add multiple more indiceis
  return img.addBands([ndvi,ndmi,evi, swir_ratio, msavi,nbr1,nbr2])

def getSentinel(aoi, start, end, cloudTresh = 80, clearTresh = 0.60):
    """
    Retrieves a Sentinel-2 image collection for a specified Area of Interest (AOI)
    within a date range, applies cloud filtering, and adds additional bands for
    further analysis.

    Parameters:
    aoi (ee.Geometry or ee.Feature): The Area of Interest (AOI) for which the
                                     Sentinel-2 images will be retrieved.
    start (str): The start year for filtering the images. Should be in 'YYYY' format.
    end (str): The end year for filtering the images. Should be in 'YYYY' format.
    cloudTresh (float): The threshold for filtering images based on the percentage
                        of cloud cover. Images with cloud coverage below this
                        percentage are selected.
    clearTresh (float): The threshold for filtering based on the cloud score. Used to create a mask.

    Returns:
    ee.ImageCollection: A processed and filtered collection of Sentinel-2 images
                        for the specified AOI and date range, with daily mosaics.
    """

    csPlus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')

    Sentinel = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                .filterBounds(aoi)
                .filterDate(f'{start}-03-01', f'{end}-11-01')
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloudTresh))
                .linkCollection(csPlus, ['cs_cdf'])
                .map(lambda img:
                     img.addBands(img.select('cs_cdf').gte(clearTresh).rename('mask')))
                .select(originalNames, commonNames)
                .map(addVariables)
    )
    # Convert the filtered and processed Sentinel collection to daily mosaics.
    Sentinel = daily_mosaics(Sentinel)
    # Return the final processed Sentinel-2 image collection.
    return Sentinel

def Reducer1(Sentinel):
    # Define reducers
    median        = ee.Reducer.median().unweighted()
    sd          = ee.Reducer.stdDev().unweighted()
    percentiles = ee.Reducer.percentile([10, 25, 50, 75, 90]).unweighted()
    allMetrics  = median.combine(sd, sharedInputs=True).combine(percentiles, sharedInputs=True)
    stm = Sentinel.reduce(allMetrics)
    return stm
def Reducer2(Sentinel):
    # Define reducers
    median        = ee.Reducer.median().unweighted()
    sd          = ee.Reducer.stdDev().unweighted()
    allMetrics  = median.combine(sd, sharedInputs=True)
    stm = Sentinel.reduce(allMetrics)
    return stm

def prepareSTMs(Sentinel):
    Spring = (Sentinel
        .filter(ee.Filter.calendarRange(3,5,'month'))
        .select(['NDVI','NDMI','EVI', 'SWIR_ratio', 'MSAVI'],
        ['Spring_NDVI','Spring_NDMI','Spring_EVI', 'Spring_SWIR_ratio', 'Spring_MSAVI'])
    )
    Summer = (Sentinel
        .filter(ee.Filter.calendarRange(6,8,'month'))
        .select(['NDVI','NDMI','EVI', 'SWIR_ratio', 'MSAVI'],
        ['Summer_NDVI','Summer_NDMI','Summer_EVI', 'Summer_SWIR_ratio', 'Summer_MSAVI'])
    )
    Autumn = (Sentinel
        .filter(ee.Filter.calendarRange(9,10,'month'))
        .select(['NDVI','NDMI','EVI', 'SWIR_ratio', 'MSAVI'],
        ['Autumn_NDVI','Autumn_NDMI','Autumn_EVI', 'Autumn_SWIR_ratio', 'Autumn_MSAVI'])
    )
    SummerAutumn = (Sentinel
        .filter(ee.Filter.calendarRange(6,10,'month'))
        .select(['NDVI','NDMI','EVI', 'SWIR_ratio', 'MSAVI'],
        ['SummerAutumn_NDVI','SummerAutumn_NDMI','SummerAutumn_EVI', 'SummerAutumn_SWIR_ratio', 'SummerAutumn_MSAVI'])
    )
    Spring_Summer = (Sentinel
        .filter(ee.Filter.calendarRange(3,8,'month'))
        .select(['NDVI','NDMI','EVI', 'SWIR_ratio', 'MSAVI'],
        ['Spring_Summer_NDVI','Spring_Summer_NDMI','Spring_Summer_EVI', 'Spring_Summer_SWIR_ratio', 'Spring_Summer_MSAVI'])
    )

    stm = Reducer1(Sentinel)
    stms = ee.Image([stm, Reducer2(Spring), Reducer2(Summer), Reducer2(Autumn), Reducer2(SummerAutumn), Reducer2(Spring_Summer)]).multiply(100).toInt()
    return stms

First to work on the embedding fields

In [ ]:
embeddings = (ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
  .filterBounds(aoi)
  .filterDate('2024-01-01', '2025-01-01')
  .mosaic()
  .multiply(100)
  )

In [ ]:
visParams = {'min': -0.3, 'max': 0.3, 'bands': ['A01', 'A16', 'A09']};
m = geemap.Map()
m.addLayer(embeddings, visParams, 'embeddings')
m.centerObject(aoi, 4)
#m

4.2 Terrain, climate, soils, irradiation

In [ ]:
def addAuxiliary(stms):
  # Load land cover image
  landcover = ee.Image('projects/g4bproject/assets/LC_con_RF').rename('landcover')
  # Define the landcover classes we're interested in
  classes = {
        1: 'urban',
        2: 'cropland',
        3: 'forest',
        4: 'shrub',
        5: 'grass',
        6: 'bare',
        7: 'water',
        8: 'wet',
  }
  # Create masks for each land cover class
  chm_data_dict = {
    'urban': landcover.eq(1).multiply(ee.Image.constant(1)),  # Assuming 1 is urban class
    'cropland': landcover.eq(2).multiply(ee.Image.constant(1)),  # Assuming 2 is cropland
    'grassland': landcover.eq(5).multiply(ee.Image.constant(1)),  # Assuming 3 is grassland
    'forest': landcover.eq(3).multiply(ee.Image.constant(1))  # Assuming 4 is forest
  }
  # Process each land cover class
  landcover_proximity_bands = []
  for name, chm_data in chm_data_dict.items():
    # Calculate proximity metrics for each class
    chm010m_mean = chm_data.reduceNeighborhood(
      reducer=ee.Reducer.sum(),
      kernel=ee.Kernel.circle(radius=1),
      #inputWeight0"mask"  #Removed extra quote
      inputWeight="mask"
    ).rename(f'{name}_prox10m')

    chm090m_mean = chm_data.reduceNeighborhood(
      reducer=ee.Reducer.sum(),
      kernel=ee.Kernel.circle(radius=9),
      #inputWeight0"mask" #Removed extra quote
      inputWeight="mask"
    ).rename(f'{name}_prox90m')

    chm360m_mean = chm_data.reduceNeighborhood(
      reducer=ee.Reducer.sum(),
      kernel=ee.Kernel.circle(radius=36),
      #inputWeight0"mask" #Removed extra quote
      inputWeight="mask"
    ).rename(f'{name}_prox360m')
    chm500m_mean = chm_data.reduceNeighborhood(
      reducer=ee.Reducer.sum(),
      kernel=ee.Kernel.circle(radius=50),
      #inputWeight0"mask" #Removed extra quote
      inputWeight="mask"
    ).rename(f'{name}_prox500m')

    # Add to bands list
    landcover_proximity_bands.extend([chm010m_mean, chm090m_mean, chm360m_mean,chm500m_mean])

  #terrain
  image = ee.Image()
  # terrain
  # Mosaic the collection to get a single global image
  alos = ee.ImageCollection("JAXA/ALOS/AW3D30/V4_1")
  elevation = alos.select('DSM').mosaic().rename('elevation')
  slope = ee.Terrain.slope(elevation).rename('slope')
  aspect = ee.Terrain.aspect(elevation).rename('aspect')
  # TPI
  focal_mean = elevation.focalMean(5, 'square')
  tpi = elevation.subtract(focal_mean).rename('tpi').multiply(100)

  solar_dni= ee.Image('projects/earthengine-legacy/assets/projects/sat-io/open-datasets/global_solar_atlas/dni_LTAy_AvgDailyTotals').rename('solar').multiply(100) #longterm average of direct normal irradiation
  #climate taken from chelsa check  https://chelsa-climate.org/, described in technical specs: https://chelsa-climate.org/wp-admin/download-page/CHELSA_tech_specification_V2.pdf
  clim_cmi=ee.Image("projects/g4bproject/assets/climate/cmi_mean").rename('clim_cmi')# Mean monthly climate moisture index
  clim_gf5=ee.Image("projects/g4bproject/assets/climate/gdgfgd5").rename('clim_gf5')# First growing degree day above 5°C
  clim_gd5=ee.Image("projects/g4bproject/assets/climate/gdd5").rename('clim_gd5')# Growing degree days heat sum above 5°C
  clim_b01=ee.Image("projects/g4bproject/assets/climate/bio1").rename('clim_b01')# mean annual air temperature
  clim_b12=ee.Image("projects/g4bproject/assets/climate/bio12").rename('clim_b12')# bio12 annual precipitation amount
  #soils
  soil_prd=ee.Image("projects/g4bproject/assets/soils/soil_cac").rename('soil_prd')#soil calcium
  soil_dth=ee.Image("projects/g4bproject/assets/soils/soil_dth").rename('soil_dth')#soil depth

  #nightlight, intactness, canopy height
  nightlight = (ee.ImageCollection('NOAA/VIIRS/DNB/ANNUAL_V22')
              .filter(ee.Filter.date('2022-01-01', '2023-01-01'))
              .select('median')
              .first()
              .rename('nightlight')
              .multiply(100)
          )
  chm = ee.Image("users/nlang/ETH_GlobalCanopyHeight_2020_10m_v1").rename('chm')
  chm_sd = ee.Image("users/nlang/ETH_GlobalCanopyHeightSD_2020_10m_v1").rename('chm_sd').multiply(100)
  #we switching this to the 10m resolution outcome
  chm_m = (ee.ImageCollection("projects/ee-chm-eu-2019/assets/planet_chm_2019") #meta CHM https://gee-community-catalog.org/projects/meta_trees/?h=canopy#dataset-citation
              .mosaic()
              .rename('chm_m'))


  smallTrees = chm_m.gt(10).And(chm_m.lt(20)).multiply(ee.Image.constant(1))
  bigTrees = chm_m.gt(20).multiply(ee.Image.constant(1))
  #proximity to forest

  smTno_009m = smallTrees.reduceNeighborhood(reducer=ee.Reducer.sum(),kernel=ee.Kernel.circle(3),inputWeight="mask").rename('smTno_009m').multiply(100);
  smTno_045m = smallTrees.reduceNeighborhood(reducer=ee.Reducer.sum(),kernel=ee.Kernel.circle(15),inputWeight="mask").rename('smTno_045m').multiply(100);
  smTno_090m = smallTrees.reduceNeighborhood(reducer=ee.Reducer.sum(),kernel=ee.Kernel.circle(30),inputWeight="mask").rename('SmTno_090m').multiply(100);
  bgTno_009m = bigTrees.reduceNeighborhood(reducer=ee.Reducer.sum(),kernel=ee.Kernel.circle(3),inputWeight="mask").rename('bgTno_009m').multiply(100);
  bgTno_045m = bigTrees.reduceNeighborhood(reducer=ee.Reducer.sum(),kernel=ee.Kernel.circle(15),inputWeight="mask").rename('bgTno_045m').multiply(100);
  bgTno_090m = bigTrees.reduceNeighborhood(reducer=ee.Reducer.sum(),kernel=ee.Kernel.circle(30),inputWeight="mask").rename('bgTno_090m').multiply(100);

  #chm009m_mean = chm.reduceNeighborhood(reducer=ee.Reducer.mean(),kernel=ee.Kernel.circle(radius=1)).rename('prox10m').multiply(100)
  #chm090m_mean = chm.reduceNeighborhood(reducer=ee.Reducer.mean(),kernel=ee.Kernel.circle(radius=9)).rename('prox90m').multiply(100)
  #chm360m_mean = chm.reduceNeighborhood(reducer=ee.Reducer.mean(),kernel=ee.Kernel.circle(radius=36)).rename('prox360m').multiply(100)
  #chm009m_stdD = chm.reduceNeighborhood(reducer=ee.Reducer.stdDev(),kernel=ee.Kernel.circle(radius=1)).rename('prox10m_std').multiply(100)
  #chm090m_stdD = chm.reduceNeighborhood(reducer=ee.Reducer.stdDev(),kernel=ee.Kernel.circle(radius=9)).rename('prox90m_std').multiply(100)
  #chm360m_stdD = chm.reduceNeighborhood(reducer=ee.Reducer.stdDev(),kernel=ee.Kernel.circle(radius=36)).rename('prox360m_std').multiply(100)
  return stms.addBands([elevation, slope, aspect, tpi,
                        solar_dni,landcover,clim_cmi, clim_gf5, clim_gd5, clim_b01, clim_b12,
                        #soil_prd, soil_dth,
                        nightlight, chm_sd,chm_m,
                        smTno_009m, smTno_045m, smTno_090m,bgTno_009m, bgTno_045m, bgTno_090m,
                        #chm009m_mean,chm090m_mean,chm360m_mean,chm009m_stdD,chm090m_stdD,chm360m_stdD,
                        landcover_proximity_bands
                        ]).toInt()

The grassland management data

In [ ]:

#Mowing Intensity ImageCollection
mowing = ee.ImageCollection('projects/ee-simonopravil/assets/DisertationProject/Mowing/Mowing_Intensity_Alps_Carpathians')

# Countries and Subregions in Alps & Carpathians
aoi = ee.FeatureCollection('projects/ee-simonopravil/assets/LULC/Alps_Carph')

def mask_no_grass(img):
    return img.updateMask(img.select('b7').neq(0)).updateMask(img.neq(999))

def filterRegion(ic, region):
    return ic.filter(ee.Filter.stringStartsWith('system:index', region))

bandNames = ['DOY1', 'DOY2', 'DOY3','DOY4', 'DOY5', 'n_mowing']
mowing_ds = (mowing
           .map(mask_no_grass)
           .select(['b1','b6'], ['DOY1', 'n_mowing'])
           .filter(ee.Filter.stringStartsWith('system:index', 'Carpathians'))
        )

In [ ]:
def addAuxiliary2(stms, mowing_ds):
    # 1. Convert the Collection into one Image with many bands
    # If the collection has 5 images with 6 bands each,
    # this creates 1 image with 30 bands.
    mowing_img = mowing_ds.toBands()

    # 2. Now apply Image methods (unmask and cast to Integer)
    mowing_median = mowing_ds.median()
    # 3. Add these many bands to your existing stms Image
    return stms.addBands([mowing_img, mowing_median])

In [ ]:
def resample(image):
    bandNames = image.bandNames()
    projection = image.select('EVI_median').projection()
    bands = image.select(bandNames)
    resampled = bands.reproject(crs=projection, scale=10).resample('bicubic')
    return resampled

In [ ]:
m = geemap.Map()
grid_subset = grid.filter(ee.Filter.eq('id',35))
m.addLayer(grid_subset)
m.centerObject(aoi, 4)
m

Map(center=[47.376687358003636, 22.726732647051985], controls=(WidgetControl(options=['position', 'transparent…

this makes a simp

An optional part::: Combining the indices towards a stacked image of multiple bands (10m resolution) and exporting a selected tile

In [ ]:
#which grid number to considered?
selGrid= 151
grid_subset = grid45.filter(ee.Filter.eq('id', selGrid))
s2 = getSentinel(grid_subset, start, end)
#print(s2.size().getInfo())
predictors = prepareSTMs(s2).clip(grid_subset.geometry()).toInt()
#predictors = addAuxiliary(predictors).updateMask(grassMask).clip(grid_subset.geometry()).toInt()
predictors_res = addAuxiliary(predictors).clip(grid_subset.geometry()).toInt()
predictors_res = addAuxiliary2(predictors_res,mowing_ds).clip(grid_subset.geometry()).toInt()
predictors_res = resample(predictors_res).updateMask(grassMask).toInt()
#print(predictors_res.getInfo())

task = ee.batch.Export.image.toDrive(
    image = predictors_res,#for now only the satellite indices are exported
    description='grid_524_eva_Alps_all_parameters_df_geo_fullsample',
    folder = 'geeDrive',
    scale = 10,
    region = grid_subset.geometry(),
     maxPixels = 1e13
 )
#task.start()

In [ ]:
#make one for the embedding fields
emb=embeddings.clip(grid_subset.geometry()).updateMask(grassMask).toInt()
task = ee.batch.Export.image.toDrive(
    image = emb,#for now only the satellite indices are exported
    description='grid_151_eva_Carps_embeddings',
    folder = 'geeDrive',
    scale = 10,
    region = grid_subset.geometry(),
     maxPixels = 1e13
 )
#task.start()

###5 Extracting the predictors at the sampled areas: tabulating, training testing data



In [ ]:
#First look if we need to delete the assets if the folder exist already:::

In [ ]:
# define a folder name
folder_ = 'projects/g4bproject/assets/evaSam_Carps' #v1 for Carps, v2 for Alps is because the later usage of the code
folder_path = f'{folder_}/'
# Attempt to get folder information, catching potential errors
try:
  ee.data.getAsset(folder_path)
  print(f'Folder exists: {folder_path}')
except ee.ee_exception.EEException as e:
  if 'Cannot find' in str(e):
    print(f'Folder does not exist: {folder_path}')
  else:
    print(f'Error checking folder {folder_path}: {e}')

Folder exists: projects/g4bproject/assets/evaSam_Carps/


If yes delete the content if necessary by the folloging code

In [ ]:
# Get a list of all assets in the folder.
assets = ee.data.listAssets({'parent': folder_path})
# Extract the asset IDs from the list.
asset_ids = [asset['id'] for asset in assets['assets']]
print(asset_ids)
# Delete each asset in the foldar.
for asset_id in asset_ids:
  try:
    ee.data.deleteAsset(asset_id)
    print(f'Deleted asset: {asset_id}')
  except Exception as e:
    print(f'Error deleting asset {asset_id}: {e}')
# Optionally, delete the empty folder itself.
try:
  ee.data.deleteAsset(folder_path)
  print(f'Deleted folder: {folder_path}')
except Exception as e:
  print(f'Error deleting folder {folder_path}: {e}')

['projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_103', 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_104', 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_112', 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_113', 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_121', 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_123', 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_131', 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_132', 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_133', 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_141', 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_142', 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_144', 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_147', 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_152', 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_153', 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_154', 'projects/g4bproject/assets/evaSam_Carp

In [ ]:
ee.data.createFolder(folder_)

{'type': 'FOLDER',
 'name': 'projects/g4bproject/assets/evaSam_Carps',
 'id': 'projects/g4bproject/assets/evaSam_Carps',
 'updateTime': '2026-03-10T14:27:11.286297Z'}

In [ ]:
print(gridID)

[6, 7, 28, 37, 38, 39, 47, 57, 77, 83, 104, 105]


In [ ]:
df=None
for grid_id in gridID:
    # Filter the grid for the current ID
    grid_cell = grid.filter(ee.Filter.eq('id', grid_id))
    botdata_clipped = botdata_raw.filterBounds(grid_cell.geometry())
    #print(botdata_clipped)
    num_data_points = botdata_clipped.size().getInfo()
    if num_data_points == 0:
        print('NODATAPOINTS')
        pass
    else:
        grid_cell = grid.filter(ee.Filter.eq('id', grid_id))
        botdata_clipped = botdata_raw.filterBounds(grid_cell.geometry())
        print(f"Grid ID: {grid_id}, N. of points: {num_data_points}")
        s2 = getSentinel(grid_cell, start, end)
        #print(s2.size().getInfo())
        predictors = prepareSTMs(s2)
        #predictors = predictors.addBands(embeddings).clip(grid_cell.geometry()).toInt()
        predictors = addAuxiliary(predictors).clip(grid_cell.geometry()).toInt()
        predictors = addAuxiliary2(predictors,mowing_ds)
        #predictors =resample(predictors).toInt()
        #predictors = addInteract(predictors).clip(grid_cell.geometry()).toInt()
        #prompt: print the s2 bands but from the  output of s2 = getSentinel(grid_cell, start, end)
        samples = predictors.unmask(0).sampleRegions(
          collection=ee.FeatureCollection(botdata_clipped)          ,
          properties= ['OBJECTID','rich_har'],
          scale=10,
          tileScale=16,
          geometries=True
          )
        # Instead of directly converting to pandas DataFrame, export to an asset
        task = ee.batch.Export.table.toAsset(
          collection=samples,
          description=f'eva_sam_grid_{grid_id}',
          assetId=f'{folder_path}eva_sam_grid_{grid_id}'  # Replace with your asset ID
          )
        task.start()
        print(f'Exporting samples for grid {grid_id} to asset...')

NODATAPOINTS
NODATAPOINTS
NODATAPOINTS
Grid ID: 102, N. of points: 113
Exporting samples for grid 102 to asset...
Grid ID: 103, N. of points: 62
Exporting samples for grid 103 to asset...
Grid ID: 104, N. of points: 39
Exporting samples for grid 104 to asset...
NODATAPOINTS
Grid ID: 112, N. of points: 95
Exporting samples for grid 112 to asset...
Grid ID: 113, N. of points: 1
Exporting samples for grid 113 to asset...
NODATAPOINTS
Grid ID: 121, N. of points: 207
Exporting samples for grid 121 to asset...
Grid ID: 122, N. of points: 642
Exporting samples for grid 122 to asset...
Grid ID: 123, N. of points: 104
Exporting samples for grid 123 to asset...
NODATAPOINTS
Grid ID: 131, N. of points: 98
Exporting samples for grid 131 to asset...
Grid ID: 132, N. of points: 87
Exporting samples for grid 132 to asset...
Grid ID: 133, N. of points: 18
Exporting samples for grid 133 to asset...
NODATAPOINTS
Grid ID: 141, N. of points: 67
Exporting samples for grid 141 to asset...
Grid ID: 142, N. o

We need to first ensure that the grid samples are all processed and stored in the cloud. It takes some time, please check the status at https://console.cloud.google.com/e

In [ ]:
assets = ee.data.listAssets({'parent': folder_path})
print(assets)
asset_ids = [asset['id'] for asset in assets['assets']]
# Print the asset IDs
for asset_id in asset_ids:
    print(asset_id)

{'assets': [{'type': 'TABLE', 'name': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_103', 'id': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_103', 'updateTime': '2026-03-04T14:23:53.264238Z'}, {'type': 'TABLE', 'name': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_104', 'id': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_104', 'updateTime': '2026-03-04T14:26:40.731842Z'}, {'type': 'TABLE', 'name': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_112', 'id': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_112', 'updateTime': '2026-03-04T14:28:56.540112Z'}, {'type': 'TABLE', 'name': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_113', 'id': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_113', 'updateTime': '2026-03-04T14:30:50.182918Z'}, {'type': 'TABLE', 'name': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_121', 'id': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_121', 'updateTime': '2026-03-04T14:37:01.096225Z'}

In [ ]:
assets = ee.data.listAssets({'parent': folder_path})
print(assets)
asset_ids = [asset['id'] for asset in assets['assets']]
#import the samples from assets
df = None
combined_fc = ee.FeatureCollection([])
for asset_id in asset_ids:
    # Import the asset
    imported_asset = ee.FeatureCollection(asset_id)
    print(f'Imported asset: {asset_id}, number of features: {imported_asset.size().getInfo()}')
    combined_fc = combined_fc.merge(imported_asset)

{'assets': [{'type': 'TABLE', 'name': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_103', 'id': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_103', 'updateTime': '2026-03-04T14:23:53.264238Z'}, {'type': 'TABLE', 'name': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_104', 'id': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_104', 'updateTime': '2026-03-04T14:26:40.731842Z'}, {'type': 'TABLE', 'name': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_112', 'id': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_112', 'updateTime': '2026-03-04T14:28:56.540112Z'}, {'type': 'TABLE', 'name': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_113', 'id': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_113', 'updateTime': '2026-03-04T14:30:50.182918Z'}, {'type': 'TABLE', 'name': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_121', 'id': 'projects/g4bproject/assets/evaSam_Carps/eva_sam_grid_121', 'updateTime': '2026-03-04T14:37:01.096225Z'}

In [ ]:
import datetime
current_date = datetime.datetime.now().strftime('%Y%m%d')
task = ee.batch.Export.table.toDrive(
            collection=combined_fc,
            description=f'richness_data_Carps_{current_date}',
            folder = 'geeDrive',
            fileFormat='csv')
task.start()


**Modelling the richnes using the Random Forests**

In [ ]:
# 1. Prepare data splits
combined_fc = combined_fc.randomColumn('random', 42)
train_split = 0.9
training_data = combined_fc.filter(ee.Filter.lt('random', train_split))
test_data = combined_fc.filter(ee.Filter.gte('random', train_split))
print("Total features in combined_fc:", combined_fc.size().getInfo())
print("Features in training_data:", training_data.size().getInfo())

# 2. Define predictor properties
#rf_properties = [
#    "elevation", "slope", "tpi", "solar", "aspect", "nightlight"]
# Get all property names from the first feature in the training set
all_names = training_data.first().propertyNames()

# Filter out the target variable and metadata to get only your predictors
# Add any other strings to this list you want to ignore (e.g., 'system:index')
#['area', 'OBJECTID', 'richness', 'Latitude', 'Longitude', 'years', 'rich_har', 'system:index
# 1. Define your excluded columns as a standard Python list
excluded_cols = ['area', 'OBJECTID', 'rich_har', 'Latitude', 'Longitude', 'years', 'system:index', 'random', 'constant']
# 2. Get all names from the feature
all_names = training_data.first().propertyNames()
# 3. Use removeAll to subtract the excluded list from the total list
rf_properties = all_names.removeAll(excluded_cols)

# 4. (Optional) Check the result
print("Predictors:", rf_properties.getInfo())
# 3. Train the model (Outside the function)
rf_model = ee.Classifier.smileRandomForest(500).setOutputMode('REGRESSION').train(
    features=training_data,
    classProperty="rich_har",
    inputProperties=rf_properties
)
print("Random Forest model trained successfully on richness.")
importance = rf_model.explain().get('importance')
print(importance.getInfo())


# 4. Evaluation
test_classified = test_data.classify(rf_model, 'predicted_richness')
error = test_classified.map(lambda f: f.set('error_sq', ee.Number(f.get('rich_har')).subtract(ee.Number(f.get('predicted_richness'))).pow(2)))
mse = ee.Number(error.aggregate_mean('error_sq'))
rmse = mse.sqrt()
print(f'Evaluation Metrics: RMSE: {rmse.getInfo()}')


# 5. Define function for prediction and mapping
def predict_richness_map(model, image, properties, aoi_geometry):
    """
    Applies the model to a multi-band image and displays the results.
    """
    # Perform classification
    prediction = image.select(properties).classify(model).rename('predicted_richness')
    return prediction  # <--- MAKE SURE THIS IS HERE

# Execute prediction function
m_richness = predict_richness_map(rf_model, predictors_res, rf_properties, grid_subset)
m_richness.getInfo()


Total features in combined_fc: 1298
Features in training_data: 1186
Predictors: ['NDVI_p50', 'Autumn_MSAVI_median', 'nir_p25', 'grassland_prox500m', 'Carpathians_2023_Mowing_Intensity_DOY1', 'Carpathians_2023_Mowing_Intensity_DOY2', 'Carpathians_2023_Mowing_Intensity_DOY5', 'Spring_Summer_EVI_stdDev', 'Carpathians_2023_Mowing_Intensity_DOY3', 'SWIR_ratio_p90', 'Carpathians_2023_Mowing_Intensity_DOY4', 'Spring_SWIR_ratio_stdDev', 'Carpathians_2021_Mowing_Intensity_DOY5', 'Carpathians_2021_Mowing_Intensity_DOY4', 'urban_prox90m', 'Carpathians_2021_Mowing_Intensity_DOY3', 'Carpathians_2021_Mowing_Intensity_DOY2', 'Carpathians_2021_Mowing_Intensity_DOY1', 'R1_median', 'NBR2_p25', 'Alps_2019_Mowing_Intensity_DOY1', 'Alps_2019_Mowing_Intensity_DOY2', 'Alps_2019_Mowing_Intensity_DOY3', 'Alps_2019_Mowing_Intensity_DOY4', 'urban_prox10m', 'Alps_2019_Mowing_Intensity_DOY5', 'R3_median', 'Carpathians_2020_Mowing_Intensity_n_mowing', 'Spring_Summer_NDVI_median', 'R1_p50', 'NBR1_median', 'Summer_SW

EEException: Image.multiply: If one image has no bands, the other must also have no bands. Got 0 and 1.

In [ ]:
# write the image
task = ee.batch.Export.image.toDrive(
    image = m_richness,#for now only the satellite indices are exported
    description='m_richness',
    folder = 'geeDrive',
    scale = 10,
    region = grid_subset.geometry(),
     maxPixels = 1e13
 )
#task.start()


In [ ]:
#for EVA MODELS
namevalues= combined_fc.aggregate_array('EVA_top1').distinct()
from_list=namevalues.getInfo()
print(from_list)
to_list = list(range(1, namevalues.size().getInfo()+1))#making it +1 to avoid 0 in the classes
print(to_list)


#(only optional) print the samples in selected tile  only
grid_subset = grid.filter(ee.Filter.eq('id', selGrid))

botdata_seltile = botdata_sp.filterBounds(grid_subset.geometry())
presentvalues= botdata_seltile.aggregate_array('EVA_top1').distinct().getInfo()
print(presentvalues)

['R35', 'R37', 'R22', 'R55', 'R44', 'R16', 'R1A', 'R36', 'R43', 'R41', 'R56', 'R1M', 'R23', 'R21', 'R51', 'R63', 'R13', 'R18', 'R52', 'R54', 'R1B', 'R1P', 'R12', 'R31', 'R57', 'R1F', 'R1E', 'R1R', 'R11']
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]


NameError: name 'selGrid' is not defined

In [ ]:
combined_fc = combined_fc.remap(from_list, to_list, 'EVA_top1')
combined_fc = combined_fc.randomColumn(columnName='random', seed=42)
task = ee.batch.Export.table.toDrive(
            collection=combined_fc,
            description='combined_fc',
            folder = 'geeDrive',
            fileFormat='csv')
task.start()
train_split = 0.8
training = combined_fc.filter(ee.Filter.lt('random', train_split))
test = combined_fc.filter(ee.Filter.gte('random', train_split))

In [ ]:
properties = [
    'elevation',
    'slope',
    'tpi',
    'solar',
    'aspect',
    'nightlight',
    'clim_b01',
    'clim_cmi',
    'clim_gd5',
    'clim_gf5',
    'clim_b12',
    'summer',
    'spring',
    'autumn',
    'EVI_median',
    'soil_dth',
    'soil_dth_1',
    'chm',
    'chm_sd'
 ]

##5 Model fitting and prediction


In [ ]:
#function to develop the models
def classify_and_evaluate(train_sample, test_sample, classifier, names_clas, properties):
    # Initialize a dictionary to store results
    results_dict = {}

    # Iterate over classifiers and their corresponding names
    for clas, name_clas in enumerate(names_clas):
        # Train the classifier
        model = classifier[clas].train(train_sample, 'EVA_top1', properties)

        # Classify the test sample
        classified_test = test_sample.classify(model)

        # Compute the error matrix and derive metrics
        test_accuracy = classified_test.errorMatrix('EVA_top1', 'classification')
        kappa = test_accuracy.kappa()
        weight = ee.Number(kappa.divide(ee.Number(1).subtract(kappa))).log10()

        # Store results in the dictionary
        results_dict[name_clas] = {
            "model": model,
            "kappa": kappa,
            "weight": weight
        }

    return results_dict

def fit_predict(tile, classifiers, clasi_name):
    """
    """

    # Initialize an empty list to hold the classified bands
    classified_bands = []


    # Iterate over the classifier names
    for clasi in clasi_name:
        # Retrieve the stored model for the classifier
        model = classifiers.get(clasi, {}).get("model", None)
        weights = classifiers.get(clasi, {}).get("weight", None)

        if model:
            # Classify the tile using the retrieved model
            classified_band = tile.classify(model).set('weight', weights).set('img_id', clasi)
            classified_bands.append(classified_band)  # Rename band to the classifier name
        else:
            print(f"Model for classifier {clasi} not found.")

    # Combine the classified bands into a single image
    base_clasi_ic = ee.ImageCollection.fromImages(classified_bands)

    return base_clasi_ic

def fit_export(tile, tile_number, classifiers, clasi_name):
    """
    """

    # Initialize an empty list to hold the classified bands
    classified_bands = []


    # Iterate over the classifier names
    for clasi in clasi_name:
        # Retrieve the stored model for the classifier
        model = classifiers.get(clasi, {}).get("model", None)
        weights = classifiers.get(clasi, {}).get("weight", None)
        kappa = classifiers.get(clasi, {}).get("kappa", None)

        classified_tile = (tile
                            .classify(model)
                            .set('img_id', clasi)
                            .set('weight', weights)
                            .set('kappa', kappa)
        )
        task = ee.batch.Export.image.toAsset(
            image = classified_tile,
            description=f'classified_{clasi}_{tile_number}',
            assetId = f'projects/g4bproject/assets/Predictors_v3/BaseModels/{clasi}_{tile_number}',
            scale = 10,
            region = grid_subset.geometry(),
            crs = 'EPSG:3035',
            maxPixels = 1e13
        )
        task.start()

def ic_weight(cl_list, based_classifier, num_classifiers, classinames):


    def map_function(classi, cl):
        # Filter the based classifier by the image ID and select the first match
        based_clas = based_classifier.filter(ee.Filter.eq('img_id', classi)).first()

        # Create a filter for the specific class (cl) and multiply by weight
        class_filter = based_clas.eq(ee.Image.constant(cl))
        weight_cls = class_filter.multiply(ee.Image.constant(based_clas.get('weight')))

        # Return the weighted class filter as a float image
        return weight_cls.float()

    # Iterate over each class in the class list (cl_list)
    img_weights = ee.ImageCollection(cl_list
                                     .map(lambda cl:ee.ImageCollection(ee.List.sequence(0, num_classifiers - 1)
                                     .map(lambda classi: map_function(classi, cl))).sum().addBands(ee.Image.constant(cl).toByte())
    ))

    # Return the collection of weighted images (one for each class in cl_list)
    return img_weights

def get_baseClassImages(tile_number, names_clas):
    # Initialize an empty list to store images
    image_list = []

    # Iterate over each class name
    for clasi in names_clas:
        # Create an ImageCollection for each class and tile
        image = ee.Image(f'projects/g4bproject/assets/Predictors_v3/BaseModels/{clasi}_{tile_number}')
        # Add the first image of the collection to the list (or any other logic)
        image_list.append(image)

    # Combine all the images into a single ImageCollection
    combined_ic = ee.ImageCollection.fromImages(image_list)

    return combined_ic

In [ ]:
num_classifiers = 4;
names_clas = ['CART', 'GB','BAYES','RF']
cl_list = ee.List(to_list)
num_clasi = ee.List.sequence(0, num_classifiers-1)
classifiers = [ee.Classifier.smileRandomForest(300).train(training, 'EVA_top1', properties)
]

classifiers = [ee.Classifier.smileCart(100),
	ee.Classifier.smileGradientTreeBoost(300),
	ee.Classifier.smileNaiveBayes(),
	ee.Classifier.smileRandomForest(300)
]
classing=classify_and_evaluate(training, test, classifiers,names_clas, properties)


In [ ]:
base_classifier = fit_export(predictors_res, 3, classing, names_clas)

In [ ]:
#the ensemble part needs to be fixed due to the computeation memory error
num_classifiers = 3;
names_clas = ['CART','BAYES','RF']
cl_list = ee.List(to_list)
num_clasi = ee.List(names_clas)
classifiers = [ee.Classifier.smileRandomForest(300).train(training, 'EVA_top1', properties)
]

classifiers = [ee.Classifier.smileCart(100),
	ee.Classifier.smileGradientTreeBoost(300),
	ee.Classifier.smileNaiveBayes(),
	ee.Classifier.smileRandomForest(300)
]
classing=classify_and_evaluate(training, test, classifiers,names_clas, properties)
base_classifier = fit_predict(predictors_res, classing, names_clas)
names_clas = ee.List(['CART','BAYES','RF'])

def ic_weight(cl):
  def map_function(classi):
      based_clas = base_classifier.filter(ee.Filter.eq('img_id', classi)).first()
      clas_filter = based_clas.eq(ee.Image.constant(cl))
      weight_cls = clas_filter.multiply(ee.Image.constant(based_clas.get('weight')))
      return weight_cls.float()

  img_weight = ee.ImageCollection(names_clas.map(map_function))
  img_weight = img_weight.sum()
  return img_weight.addBands(ee.Image.constant(cl).toByte())

ic_weight_cls = cl_list.map(ic_weight)
enseble_class = ee.ImageCollection(ic_weight_cls).reduce(ee.Reducer.max(2)).rename(['weight','ensemble_cls'])

task = ee.batch.Export.image.toAsset(
    image = enseble_class,
    description='enseble_class_tile3_test',
    assetId = 'projects/g4bproject/assets/Predictors/enseble_class_tile3_test4',
    scale = 20,
    region = grid_subset.geometry(),
    maxPixels = 1e13
)
task.start()

# Task
Train an `ee.Classifier.smileRandomForest` model using the training features from the `combined_fc` FeatureCollection (split from the samples extracted across the Alps and Carpathians grids) to predict grassland species richness. Use the specified predictor properties, including "elevation", "slope", "tpi", "solar", "aspect", "nightlight", "clim_b01", "clim_cmi", "clim_gd5", "clim_gf5", "clim_b12", "summer", "spring", "autumn", "EVI_median", "soil_dth", "chm", and "chm_sd". Once trained, apply the model to the `predictors_res` image to generate a richness prediction map for the study area.

## train_rf_model

### Subtask:
Initialize and train an ee.Classifier.smileRandomForest model using the training FeatureCollection and the specified predictor properties.


**Reasoning**:
I will initialize the list of predictor properties and train the Earth Engine Random Forest classifier using the 'richness' property from the training data.



In [ ]:
# Define the predictor properties
rf_properties = [
    "elevation", "slope", "tpi", "solar", "aspect", "nightlight",
    "clim_b01", "clim_cmi", "clim_gd5", "clim_gf5", "clim_b12",
    "summer", "spring", "autumn", "EVI_median", "soil_dth",
    "chm", "chm_sd"
]

# Initialize and train the Random Forest model
# Using 300 trees as per the notebook's earlier logic for RF
rf_model = ee.Classifier.smileRandomForest(300).train(
    features=training,
    classProperty="richness",
    inputProperties=rf_properties
)

print("Random Forest model trained successfully.")


NameError: name 'ee' is not defined